<a href="https://colab.research.google.com/github/NMinarecioglu/kizilirmak-drought-propagation/blob/main/Copula_and_Bayesian_Network_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 pip install pandas numpy openpyxl

In [ ]:
"""
Copula Analysis for Drought Events
Fits bivariate copulas to (Duration, Severity) and (Duration, Peak) pairs
for each Station / Index / Scale / Threshold combination.

Requirements:
    pip install pandas numpy matplotlib scipy statsmodels openpyxl

Output:
    copula_plots/   — scatter + copula density plots
    copula_results.xlsx — fitted parameters and goodness-of-fit
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as mgrid
from scipy import stats
from scipy.optimize import minimize_scalar, minimize
from scipy.stats import kendalltau, spearmanr
import os, warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
INPUT_FILE = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Run_Theory_NEW/drought_events.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Copula_Results"

# Filter settings
THRESHOLD  = -0.5          # Use this threshold for copula fitting
PAIRS      = [("Duration", "Severity"), ("Duration", "Peak"), ("Severity", "Peak")]
MIN_EVENTS = 10            # Minimum events required to fit a copula
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

df_all = pd.read_csv(INPUT_FILE, parse_dates=["Start", "End"])
df_all["Peak"] = df_all["Peak"].abs()   # Work with absolute peak values

df = df_all[df_all["Threshold"] == THRESHOLD].copy().reset_index(drop=True)
print(f"Total events at threshold={THRESHOLD}: {len(df)}")


# ===========================================================
# EMPIRICAL CDF (Weibull plotting position)
# ===========================================================
def empirical_cdf(x):
    n    = len(x)
    rank = stats.rankdata(x, method="average")
    return rank / (n + 1)   # Weibull formula


# ===========================================================
# COPULA FAMILIES
# ===========================================================

# --- Gaussian ---
def gaussian_copula_loglik(rho, u, v):
    """Log-likelihood of Gaussian copula."""
    eps = 1e-10
    u   = np.clip(u, eps, 1 - eps)
    v   = np.clip(v, eps, 1 - eps)
    x   = stats.norm.ppf(u)
    y   = stats.norm.ppf(v)
    det = 1 - rho ** 2
    ll  = -0.5 * np.log(det) - (rho**2 * (x**2 + y**2) - 2*rho*x*y) / (2*det)
    return np.sum(ll)

def fit_gaussian(u, v):
    tau, _ = kendalltau(u, v)
    rho0   = np.sin(np.pi * tau / 2)
    res    = minimize_scalar(
        lambda r: -gaussian_copula_loglik(r, u, v),
        bounds=(-0.999, 0.999), method="bounded"
    )
    rho = res.x
    ll  = gaussian_copula_loglik(rho, u, v)
    return {"family": "Gaussian", "param1": round(rho, 4), "param2": np.nan,
            "loglik": round(ll, 4)}

# --- Clayton ---
def clayton_copula_cdf(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    return np.maximum(u**(-theta) + v**(-theta) - 1, eps) ** (-1/theta)

def clayton_copula_pdf(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    A   = u**(-theta) + v**(-theta) - 1
    pdf = (1+theta) * (u*v)**(-(1+theta)) * A**(-(2+1/theta))
    return np.maximum(pdf, eps)

def fit_clayton(u, v):
    tau, _ = kendalltau(u, v)
    theta0 = max(2*tau / (1-tau), 0.01)
    res    = minimize_scalar(
        lambda t: -np.sum(np.log(np.maximum(clayton_copula_pdf(u, v, t), 1e-300))),
        bounds=(0.01, 50), method="bounded"
    )
    theta = res.x
    ll    = np.sum(np.log(np.maximum(clayton_copula_pdf(u, v, theta), 1e-300)))
    return {"family": "Clayton", "param1": round(theta, 4), "param2": np.nan,
            "loglik": round(ll, 4)}

# --- Gumbel ---
def gumbel_copula_cdf(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    A   = (-np.log(u))**theta + (-np.log(v))**theta
    return np.exp(-A**(1/theta))

def gumbel_copula_pdf(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    lu  = -np.log(u)
    lv  = -np.log(v)
    A   = lu**theta + lv**theta
    C   = np.exp(-A**(1/theta))
    pdf = C * (u*v)**(-1) * A**(-2+2/theta) * (lu*lv)**(theta-1) * \
          (1 + (theta-1)*A**(-1/theta))
    return np.maximum(pdf, eps)

def fit_gumbel(u, v):
    tau, _ = kendalltau(u, v)
    theta0 = max(1 / (1 - tau), 1.01)
    res    = minimize_scalar(
        lambda t: -np.sum(np.log(np.maximum(gumbel_copula_pdf(u, v, t), 1e-300))),
        bounds=(1.001, 50), method="bounded"
    )
    theta = res.x
    ll    = np.sum(np.log(np.maximum(gumbel_copula_pdf(u, v, theta), 1e-300)))
    return {"family": "Gumbel", "param1": round(theta, 4), "param2": np.nan,
            "loglik": round(ll, 4)}

# --- Frank ---
def frank_copula_pdf(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    et  = np.exp(-theta)
    eu  = np.exp(-theta*u)
    ev  = np.exp(-theta*v)
    num = -theta * (et-1) * eu * ev
    den = ((et-1) + (eu-1)*(ev-1))**2
    pdf = num / np.maximum(den, eps)
    return np.maximum(np.abs(pdf), eps)

def fit_frank(u, v):
    res = minimize_scalar(
        lambda t: -np.sum(np.log(np.maximum(frank_copula_pdf(u, v, t), 1e-300))),
        bounds=(-50, 50), method="bounded"
    )
    theta = res.x
    ll    = np.sum(np.log(np.maximum(frank_copula_pdf(u, v, theta), 1e-300)))
    return {"family": "Frank", "param1": round(theta, 4), "param2": np.nan,
            "loglik": round(ll, 4)}


# ===========================================================
# SELECT BEST COPULA (AIC)
# ===========================================================
def select_best_copula(u, v):
    results = []
    for fit_fn in [fit_gaussian, fit_clayton, fit_gumbel, fit_frank]:
        try:
            r   = fit_fn(u, v)
            k   = 1   # all have 1 parameter
            aic = 2*k - 2*r["loglik"]
            r["AIC"] = round(aic, 4)
            results.append(r)
        except Exception as e:
            pass
    if not results:
        return None
    results.sort(key=lambda x: x["AIC"])
    return results   # sorted best first


# ===========================================================
# JOINT EXCEEDANCE PROBABILITY
# ===========================================================
def joint_exceed_prob(u, v, copula_info):
    """P(U > u_med, V > v_med) using best copula."""
    u_med = np.median(u)
    v_med = np.median(v)
    family = copula_info["family"]
    p      = copula_info["param1"]
    eps    = 1e-10

    if family == "Gaussian":
        from scipy.stats import multivariate_normal
        rho   = p
        cov   = [[1, rho], [rho, 1]]
        x_med = stats.norm.ppf(u_med)
        y_med = stats.norm.ppf(v_med)
        C_med = multivariate_normal.cdf([x_med, y_med], cov=cov)
    elif family == "Clayton":
        C_med = clayton_copula_cdf(np.array([u_med]), np.array([v_med]), p)[0]
    elif family == "Gumbel":
        C_med = gumbel_copula_cdf(np.array([u_med]), np.array([v_med]), p)[0]
    elif family == "Frank":
        et  = np.exp(-p)
        eu  = np.exp(-p*u_med)
        ev  = np.exp(-p*v_med)
        C_med = -1/p * np.log(1 + (eu-1)*(ev-1)/np.maximum(et-1, eps))
    else:
        C_med = u_med * v_med

    # P(U>u_med, V>v_med) = 1 - u_med - v_med + C(u_med, v_med)
    joint_p = 1 - u_med - v_med + C_med
    return round(float(joint_p), 4)


# ===========================================================
# PLOT: Scatter + Copula density
# ===========================================================
def plot_copula(u, v, x_raw, y_raw, best, station, idx, scale, pair, out_path):
    x_lab, y_lab = pair
    fig = plt.figure(figsize=(13, 5))
    fig.patch.set_facecolor("white")
    gs  = mgrid.GridSpec(1, 3, figure=fig, wspace=0.35)

    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    ax3 = fig.add_subplot(gs[2])

    # --- Raw scatter ---
    ax1.scatter(x_raw, y_raw, alpha=0.6, s=20, color="#1f6f2e", edgecolors="white", lw=0.3)
    ax1.set_xlabel(x_lab, fontsize=10)
    ax1.set_ylabel(y_lab, fontsize=10)
    ax1.set_title("Observed Data", fontsize=10, fontweight="bold")
    ax1.grid(linestyle="--", alpha=0.4)

    # --- Uniform margin scatter ---
    ax2.scatter(u, v, alpha=0.6, s=20, color="#084594", edgecolors="white", lw=0.3)
    ax2.set_xlabel(f"F({x_lab})", fontsize=10)
    ax2.set_ylabel(f"F({y_lab})", fontsize=10)
    ax2.set_title("Probability Integral Transform", fontsize=10, fontweight="bold")
    ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
    ax2.grid(linestyle="--", alpha=0.4)

    # --- Copula density contour ---
    ug, vg = np.meshgrid(np.linspace(0.01, 0.99, 80),
                          np.linspace(0.01, 0.99, 80))
    family = best["family"]
    p      = best["param1"]

    try:
        if family == "Gaussian":
            rho  = p
            xg   = stats.norm.ppf(ug)
            yg   = stats.norm.ppf(vg)
            det  = 1 - rho**2
            dens = np.exp(-0.5*(rho**2*(xg**2+yg**2)-2*rho*xg*yg)/det) / np.sqrt(det)
        elif family == "Clayton":
            dens = (1+p)*(ug*vg)**(-(1+p)) * \
                   np.maximum(ug**(-p)+vg**(-p)-1, 1e-10)**(-(2+1/p))
        elif family == "Gumbel":
            dens = gumbel_copula_pdf(ug, vg, p)
        elif family == "Frank":
            dens = frank_copula_pdf(ug, vg, p)
        else:
            dens = np.ones_like(ug)

        dens = np.where(np.isfinite(dens) & (dens > 0), dens, 1e-10)
        cf   = ax3.contourf(ug, vg, dens, levels=15, cmap="RdYlGn_r")
        ax3.contour(ug, vg, dens, levels=8, colors="black", linewidths=0.4, alpha=0.5)
        plt.colorbar(cf, ax=ax3, label="Density", pad=0.02)
    except Exception:
        ax3.text(0.5, 0.5, "Density\nunavailable", ha="center", va="center",
                 transform=ax3.transAxes)

    ax3.scatter(u, v, alpha=0.4, s=10, color="white", edgecolors="black", lw=0.4)
    ax3.set_xlabel(f"F({x_lab})", fontsize=10)
    ax3.set_ylabel(f"F({y_lab})", fontsize=10)
    ax3.set_title(f"{family} Copula Density\nθ={p}  AIC={best['AIC']}",
                  fontsize=10, fontweight="bold")
    ax3.set_xlim(0, 1); ax3.set_ylim(0, 1)

    tau, _  = kendalltau(u, v)
    rho_s,_ = spearmanr(u, v)
    fig.suptitle(
        f"{station} — {idx}-{scale}  |  {x_lab} vs {y_lab}\n"
        f"Kendall τ={tau:.3f}  Spearman ρ={rho_s:.3f}  "
        f"Best: {family} (AIC={best['AIC']})",
        fontsize=11, fontweight="bold", y=1.02
    )

    plt.tight_layout()
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close()


# ===========================================================
# MAIN LOOP
# ===========================================================
print("=" * 60)
print("  Copula Analysis — Starting")
print("=" * 60 + "\n")

records = []

for station in sorted(df["Station"].unique()):
    for idx in ["SPI", "SDI", "RDI"]:
        for scale in [3, 6, 12]:
            sub = df[
                (df["Station"] == station) &
                (df["Index"]   == idx) &
                (df["Scale"]   == scale)
            ].dropna(subset=["Duration", "Severity", "Peak"]).copy()

            if len(sub) < MIN_EVENTS:
                print(f"  [SKIP] {station} {idx}-{scale}: only {len(sub)} events")
                continue

            print(f">>> {station} {idx}-{scale}  ({len(sub)} events)")

            for pair in PAIRS:
                x_col, y_col = pair
                x_raw = sub[x_col].values.astype(float)
                y_raw = sub[y_col].values.astype(float)

                # Uniform margins (empirical CDF)
                u = empirical_cdf(x_raw)
                v = empirical_cdf(y_raw)

                # Kendall tau
                tau, tau_p = kendalltau(x_raw, y_raw)

                # Fit all copulas and select best
                all_fits = select_best_copula(u, v)
                if all_fits is None:
                    continue
                best = all_fits[0]

                # Joint exceedance probability
                jp = joint_exceed_prob(u, v, best)

                # Save plot
                tag      = f"{station}_{idx}-{scale}_{x_col}_vs_{y_col}"
                out_path = os.path.join(OUTPUT_DIR, f"Copula_{tag}.png")
                plot_copula(u, v, x_raw, y_raw, best,
                            station, idx, scale, pair, out_path)

                # Record
                rec = {
                    "Station"     : station,
                    "Index"       : idx,
                    "Scale"       : scale,
                    "Threshold"   : THRESHOLD,
                    "Pair"        : f"{x_col}-{y_col}",
                    "N_Events"    : len(sub),
                    "Kendall_tau" : round(tau, 4),
                    "Tau_pvalue"  : round(tau_p, 4),
                    "Best_Copula" : best["family"],
                    "Param"       : best["param1"],
                    "AIC_Best"    : best["AIC"],
                    "LogLik"      : best["loglik"],
                    "Joint_Exceed_Prob": jp,
                }
                # Add all family AICs
                for r in all_fits:
                    rec[f"AIC_{r['family']}"] = r["AIC"]

                records.append(rec)
                print(f"    {x_col} vs {y_col}: Best={best['family']}  "
                      f"τ={tau:.3f}  AIC={best['AIC']}")

            print()

# ===========================================================
# SAVE RESULTS
# ===========================================================
df_res = pd.DataFrame(records)
out_xlsx = os.path.join(OUTPUT_DIR, "copula_results.xlsx")
out_csv  = os.path.join(OUTPUT_DIR, "copula_results.csv")
df_res.to_excel(out_xlsx, index=False)
df_res.to_csv(out_csv,   index=False)

print("=" * 60)
print(f"  Total fits : {len(df_res)}")
print(f"  Saved → {out_xlsx}")
print(f"  Saved → {out_csv}")
print("=" * 60)

print("\nBest copula family distribution:")
print(df_res["Best_Copula"].value_counts().to_string())


In [ ]:
"""
Bayesian Network Analysis for Drought Characteristics
Uses discretized drought features (Duration, Severity, Peak) to build
a Bayesian Network and compute conditional probabilities.

Requirements:
    pip install pandas numpy matplotlib scipy pgmpy openpyxl seaborn

Output:
    Bayesian_Network/ folder:
        - bn_results.xlsx       : conditional probability tables
        - BN_Structure_*.png    : network structure plots
        - BN_Heatmap_*.png      : conditional probability heatmaps
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import os, warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
EVENTS_FILE  = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Run_Theory_NEW/drought_events.csv"
COPULA_FILE  = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Copula_Results/copula_results.xlsx"
OUTPUT_DIR   = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Bayesian_Network"

THRESHOLD    = -0.5       # Same threshold used in copula
SCALE        = 6          # Focus scale (change to 3 or 12 as needed)
INDICES      = ["SPI", "SDI", "RDI"]

# Discretization thresholds (percentile-based)
# Duration  : Short / Moderate / Long
# Severity  : Low / Moderate / High
# Peak      : Mild / Moderate / Extreme
PERCENTILES  = [33, 67]   # Tercile split
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------------------------------------
# Load data
# ----------------------------------------------------------
df_ev = pd.read_csv(EVENTS_FILE, parse_dates=["Start", "End"])
df_ev["Peak"] = df_ev["Peak"].abs()
df_ev = df_ev[df_ev["Threshold"] == THRESHOLD].copy()

df_cop = pd.read_excel(COPULA_FILE)

print(f"Events loaded : {len(df_ev)}")
print(f"Copula results: {len(df_cop)}\n")


# ----------------------------------------------------------
# Discretize continuous variables into 3 categories
# ----------------------------------------------------------
def discretize(series, labels):
    p1, p2 = np.percentile(series, PERCENTILES)
    cats = pd.cut(series, bins=[-np.inf, p1, p2, np.inf],
                  labels=labels, include_lowest=True)
    return cats, p1, p2

DUR_LABELS = ["Short", "Moderate", "Long"]
SEV_LABELS = ["Low",   "Moderate", "High"]
PEK_LABELS = ["Mild",  "Moderate", "Extreme"]


# ----------------------------------------------------------
# Conditional Probability Tables (manual Bayesian Network)
# Structure: Duration → Severity → Peak
#            Duration → Peak
# ----------------------------------------------------------
def compute_cpt(df_sub):
    """
    Computes P(Severity | Duration) and P(Peak | Severity, Duration)
    Returns dict of DataFrames.
    """
    results = {}

    # P(Duration)
    p_dur = df_sub["D_cat"].value_counts(normalize=True).reindex(DUR_LABELS, fill_value=0)
    results["P_Duration"] = p_dur.round(4)

    # P(Severity | Duration)
    cpt_sev = pd.crosstab(df_sub["D_cat"], df_sub["S_cat"], normalize="index")
    cpt_sev = cpt_sev.reindex(index=DUR_LABELS, columns=SEV_LABELS, fill_value=0).round(4)
    results["P_Severity_given_Duration"] = cpt_sev

    # P(Peak | Severity)
    cpt_pek = pd.crosstab(df_sub["S_cat"], df_sub["P_cat"], normalize="index")
    cpt_pek = cpt_pek.reindex(index=SEV_LABELS, columns=PEK_LABELS, fill_value=0).round(4)
    results["P_Peak_given_Severity"] = cpt_pek

    # P(Peak | Duration, Severity)  — joint conditioning
    cpt_joint = df_sub.groupby(["D_cat", "S_cat"])["P_cat"] \
                      .value_counts(normalize=True) \
                      .unstack(fill_value=0) \
                      .reindex(columns=PEK_LABELS, fill_value=0).round(4)
    results["P_Peak_given_Duration_Severity"] = cpt_joint

    # Joint exceedance: P(D=Long, S=High, P=Extreme)
    n = len(df_sub)
    p_extreme = ((df_sub["D_cat"]=="Long") & (df_sub["S_cat"]=="High") &
                 (df_sub["P_cat"]=="Extreme")).sum() / n
    results["P_Extreme_Joint"] = round(p_extreme, 4)

    # Trigger probabilities: P(S=High | D=Long)
    sub_long = df_sub[df_sub["D_cat"] == "Long"]
    p_sev_high_given_long = (sub_long["S_cat"] == "High").mean() if len(sub_long) > 0 else np.nan
    results["P_SevHigh_given_DurLong"] = round(p_sev_high_given_long, 4)

    # P(Peak=Extreme | S=High)
    sub_high = df_sub[df_sub["S_cat"] == "High"]
    p_ext_given_high = (sub_high["P_cat"] == "Extreme").mean() if len(sub_high) > 0 else np.nan
    results["P_PeakExtreme_given_SevHigh"] = round(p_ext_given_high, 4)

    return results


# ----------------------------------------------------------
# Plot: Network Structure
# ----------------------------------------------------------
def plot_bn_structure(station, idx, scale, cpt_results, out_path):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.patch.set_facecolor("white")

    # Colors per index
    colors = {"SPI": "#1f6f2e", "SDI": "#084594", "RDI": "#8c2d04"}
    col    = colors.get(idx, "gray")

    # --- Panel 1: P(Severity | Duration) ---
    ax = axes[0]
    cpt = cpt_results["P_Severity_given_Duration"]
    x   = np.arange(len(DUR_LABELS))
    w   = 0.25
    shades = ["#a8d5a2", "#4caf50", "#1b5e20"]
    for i, sev in enumerate(SEV_LABELS):
        vals = cpt[sev].values if sev in cpt.columns else np.zeros(3)
        ax.bar(x + i*w, vals, w, label=sev, color=shades[i], edgecolor="white")
    ax.set_xticks(x + w)
    ax.set_xticklabels(DUR_LABELS, fontsize=9)
    ax.set_ylabel("Probability", fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_title("P(Severity | Duration)", fontsize=10, fontweight="bold")
    ax.legend(fontsize=8, title="Severity")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    # --- Panel 2: P(Peak | Severity) ---
    ax = axes[1]
    cpt = cpt_results["P_Peak_given_Severity"]
    shades2 = ["#ffcc80", "#ef6c00", "#b71c1c"]
    for i, pek in enumerate(PEK_LABELS):
        vals = cpt[pek].values if pek in cpt.columns else np.zeros(3)
        ax.bar(x + i*w, vals, w, label=pek, color=shades2[i], edgecolor="white")
    ax.set_xticks(x + w)
    ax.set_xticklabels(SEV_LABELS, fontsize=9)
    ax.set_ylabel("Probability", fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_title("P(Peak | Severity)", fontsize=10, fontweight="bold")
    ax.legend(fontsize=8, title="Peak")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    # --- Panel 3: Key probabilities (trigger thresholds) ---
    ax = axes[2]
    ax.axis("off")

    p_ext   = cpt_results["P_Extreme_Joint"]
    p_sh_dl = cpt_results["P_SevHigh_given_DurLong"]
    p_pe_sh = cpt_results["P_PeakExtreme_given_SevHigh"]
    p_dur   = cpt_results["P_Duration"]

    table_data = [
        ["P(Duration=Short)",            f"{p_dur.get('Short', 0):.3f}"],
        ["P(Duration=Moderate)",         f"{p_dur.get('Moderate', 0):.3f}"],
        ["P(Duration=Long)",             f"{p_dur.get('Long', 0):.3f}"],
        ["─────────────────────", "──────"],
        ["P(Sev=High | Dur=Long)",       f"{p_sh_dl:.3f}" if not np.isnan(p_sh_dl) else "N/A"],
        ["P(Peak=Extreme | Sev=High)",   f"{p_pe_sh:.3f}" if not np.isnan(p_pe_sh) else "N/A"],
        ["─────────────────────", "──────"],
        ["P(Long, High, Extreme)",       f"{p_ext:.3f}"],
    ]

    tbl = ax.table(
        cellText=table_data,
        colLabels=["Probability Statement", "Value"],
        loc="center", cellLoc="left"
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1.3, 1.6)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor(col)
            cell.set_text_props(color="white", fontweight="bold")
        elif r in [4, 5, 7]:
            cell.set_facecolor("#fff3e0")
    ax.set_title("Trigger Probabilities", fontsize=10, fontweight="bold")

    fig.suptitle(
        f"Bayesian Network — {station}  {idx}-{scale}\n"
        f"Structure: Duration → Severity → Peak",
        fontsize=12, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close()


# ----------------------------------------------------------
# Plot: Conditional Probability Heatmap
# ----------------------------------------------------------
def plot_heatmap(station, idx, scale, cpt_results, out_path):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.patch.set_facecolor("white")

    cpt1 = cpt_results["P_Severity_given_Duration"]
    cpt2 = cpt_results["P_Peak_given_Severity"]

    sns.heatmap(cpt1, ax=axes[0], annot=True, fmt=".3f",
                cmap="YlOrRd", vmin=0, vmax=1,
                linewidths=0.5, linecolor="white",
                cbar_kws={"label": "Probability"})
    axes[0].set_title("P(Severity | Duration)", fontsize=11, fontweight="bold")
    axes[0].set_xlabel("Severity Category", fontsize=9)
    axes[0].set_ylabel("Duration Category", fontsize=9)

    sns.heatmap(cpt2, ax=axes[1], annot=True, fmt=".3f",
                cmap="YlOrRd", vmin=0, vmax=1,
                linewidths=0.5, linecolor="white",
                cbar_kws={"label": "Probability"})
    axes[1].set_title("P(Peak | Severity)", fontsize=11, fontweight="bold")
    axes[1].set_xlabel("Peak Category", fontsize=9)
    axes[1].set_ylabel("Severity Category", fontsize=9)

    fig.suptitle(
        f"Conditional Probability Heatmaps — {station}  {idx}-{scale}",
        fontsize=12, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close()


# ----------------------------------------------------------
# Main loop
# ----------------------------------------------------------
print("=" * 60)
print("  Bayesian Network Analysis — Starting")
print("=" * 60 + "\n")

all_records = []

stations = sorted(df_ev["Station"].unique())

for station in stations:
    for idx in INDICES:
        for scale in [3, 6, 12]:
            sub = df_ev[
                (df_ev["Station"] == station) &
                (df_ev["Index"]   == idx) &
                (df_ev["Scale"]   == scale)
            ].dropna(subset=["Duration", "Severity", "Peak"]).copy()

            if len(sub) < 10:
                print(f"  [SKIP] {station} {idx}-{scale}: only {len(sub)} events")
                continue

            # Discretize
            sub["D_cat"], d1, d2 = discretize(sub["Duration"], DUR_LABELS)
            sub["S_cat"], s1, s2 = discretize(sub["Severity"], SEV_LABELS)
            sub["P_cat"], p1, p2 = discretize(sub["Peak"],     PEK_LABELS)

            print(f">>> {station} {idx}-{scale}  ({len(sub)} events)")
            print(f"    Duration thresholds : {d1:.1f} / {d2:.1f} months")
            print(f"    Severity thresholds : {s1:.2f} / {s2:.2f}")
            print(f"    Peak thresholds     : {p1:.2f} / {p2:.2f}")

            # CPT
            cpt_res = compute_cpt(sub)
            print(f"    P(Sev=High | Dur=Long)      = {cpt_res['P_SevHigh_given_DurLong']:.3f}")
            print(f"    P(Peak=Extreme | Sev=High)  = {cpt_res['P_PeakExtreme_given_SevHigh']:.3f}")
            print(f"    P(Long & High & Extreme)    = {cpt_res['P_Extreme_Joint']:.3f}\n")

            # Plots
            tag = f"{station}_{idx}-{scale}"
            plot_bn_structure(
                station, idx, scale, cpt_res,
                os.path.join(OUTPUT_DIR, f"BN_Structure_{tag}.png")
            )
            plot_heatmap(
                station, idx, scale, cpt_res,
                os.path.join(OUTPUT_DIR, f"BN_Heatmap_{tag}.png")
            )

            # Record summary
            p_dur = cpt_res["P_Duration"]
            all_records.append({
                "Station"                     : station,
                "Index"                       : idx,
                "Scale"                       : scale,
                "N_Events"                    : len(sub),
                "Dur_Short_thresh"            : round(d1, 2),
                "Dur_Long_thresh"             : round(d2, 2),
                "Sev_Low_thresh"              : round(s1, 2),
                "Sev_High_thresh"             : round(s2, 2),
                "Peak_Mild_thresh"            : round(p1, 2),
                "Peak_Extreme_thresh"         : round(p2, 2),
                "P_Duration_Short"            : p_dur.get("Short", 0),
                "P_Duration_Moderate"         : p_dur.get("Moderate", 0),
                "P_Duration_Long"             : p_dur.get("Long", 0),
                "P_SevHigh_given_DurLong"     : cpt_res["P_SevHigh_given_DurLong"],
                "P_PeakExtreme_given_SevHigh" : cpt_res["P_PeakExtreme_given_SevHigh"],
                "P_Extreme_Joint"             : cpt_res["P_Extreme_Joint"],
            })

# ----------------------------------------------------------
# Save results
# ----------------------------------------------------------
df_bn = pd.DataFrame(all_records)
out_xlsx = os.path.join(OUTPUT_DIR, "bn_results.xlsx")
out_csv  = os.path.join(OUTPUT_DIR, "bn_results.csv")
df_bn.to_excel(out_xlsx, index=False)
df_bn.to_csv(out_csv,   index=False)

print("=" * 60)
print(f"  Total combinations : {len(df_bn)}")
print(f"  Saved → {out_xlsx}")
print("=" * 60)

# Summary: highest joint extreme probabilities
print("\nTop 10 — Highest P(Long & High & Extreme):")
print(df_bn.nlargest(10, "P_Extreme_Joint")[
    ["Station", "Index", "Scale", "N_Events",
     "P_SevHigh_given_DurLong",
     "P_PeakExtreme_given_SevHigh",
     "P_Extreme_Joint"]
].to_string(index=False))